<div align="center">

  <h1><img src="https://raw.githubusercontent.com/auxeno/ion/main/assets/logo.png" alt="Ion" width="72"><br>Ion Tour Notebook</h1>

  [![Open in nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/Auxeno/ion/blob/d04b6bf/examples/ion_tour.ipynb)
  [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/auxeno/ion/blob/main/examples/ion_tour.ipynb)

</div>

---

A hands-on walkthrough of [Ion](https://github.com/auxeno/ion), the most minimal neural network library for JAX. Ion introduces the fewest abstractions needed to build, train, and manage neural networks on top of JAX's ecosystem.

In this notebook we demo the three core concepts from Ion:

1. **`Module`**: define models as immutable JAX pytrees
2. **`Param`**: mark arrays as trainable or frozen (frozen params apply `stop_gradient` automatically)
3. **`ion.Optimizer`**: wrap an optax optimizer with Param-aware updates and automatic frozen-param partitioning

Since Ion models are standard pytrees, native JAX transforms like `jax.grad`, `jax.jit`, and `jax.vmap` work directly with no wrappers.

> *Click the badges above to open in **nbviewer** (fully rendered) or **Colab** (run it yourself).*

In [1]:
# !pip install ion-nn optax plotly tqdm

In [2]:
import jax
import jax.numpy as jnp
import optax
import plotly.graph_objects as go
import plotly.io as pio

import ion
from ion import nn

pio.renderers.default = "notebook_connected"

## The dataset

We'll learn $\sin(x)$ on $[-\pi, \pi]$, a simple regression task.

In [3]:
x_train = jnp.linspace(-jnp.pi, jnp.pi, 100).reshape(-1, 1)
y_train = jnp.sin(x_train)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_train[:, 0], y=y_train[:, 0], mode="lines", name="sin(x)", line=dict(width=2, color="rgb(96,96,96)", dash="dash")))
fig.update_layout(title="Training data", xaxis_title="x", yaxis_title="y", height=400, template="plotly_white")
fig.show()

## Defining a Module

Subclass `nn.Module`, declare your fields with type annotations, and write `__init__` and `__call__`. Ion registers the class as a JAX pytree and makes it immutable after `__init__`.

We'll define a custom `Linear` layer and compose an `MLP` from it. This covers the three kinds of field you'll use:

- **`nn.Param`** (`w`, `b`): wraps an array and marks it as a trainable parameter
- **Callables** (`activation`): static metadata, invisible to JAX tracing
- **Plain arrays** (`input_scale`): part of the pytree but not a `Param`, so `Optimizer.update` leaves them unchanged

In [4]:
from collections.abc import Callable
from jaxtyping import Array, Float, PRNGKeyArray


class Linear(nn.Module):
    w: nn.Param
    b: nn.Param
    activation: Callable | None

    def __init__(self, in_dim: int, out_dim: int, activation=None, *, key: PRNGKeyArray):
        self.w = nn.Param(jax.nn.initializers.glorot_uniform()(key, (in_dim, out_dim)))
        self.b = nn.Param(jnp.zeros(out_dim))
        self.activation = activation

    def __call__(self, x):
        x = x @ self.w + self.b
        if self.activation is not None:
            x = self.activation(x)
        return x


class MLP(nn.Module):
    fc_1: Linear
    fc_2: Linear
    input_scale: jax.Array

    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, *, key: PRNGKeyArray):
        keys = jax.random.split(key)
        self.fc_1 = Linear(in_dim, hidden_dim, activation=jax.nn.relu, key=keys[0])
        self.fc_2 = Linear(hidden_dim, out_dim, key=keys[1])
        self.input_scale = jnp.array(1.0 / jnp.pi)

    def __call__(self, x: Float[Array, "... d"]) -> Float[Array, "... d"]:
        x = x * self.input_scale
        x = self.fc_1(x)
        x = self.fc_2(x)
        return x


model = MLP(in_dim=1, hidden_dim=16, out_dim=1, key=jax.random.key(0))
model

MLP(
  fc_1=Linear(
    w=Param(f32[1, 16], trainable=True),
    b=Param(f32[16], trainable=True),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=True),
    b=Param(f32[1], trainable=True),
    activation=None,
  ),
  input_scale=f32[],
)

Treescope is enabled by default for interactive rendering in notebooks. Click on elements to expand or collapse them.

Ion also has a built-in text representation for terminals and logging:

In [5]:
ion.disable_treescope()
print(model)
ion.enable_treescope()

MLP(
  fc_1=Linear(
    w=Param(f32[1, 16], trainable=True),
    b=Param(f32[16], trainable=True),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=True),
    b=Param(f32[1], trainable=True),
    activation=None,
  ),
  input_scale=f32[],
)


## Params

Every learnable array in a model is wrapped in `nn.Param`. Params track whether they are **trainable** or **frozen** and behave like regular arrays in expressions, so no unwrapping is needed.

In [6]:
model.fc_1.w

Param(f32[1, 16], trainable=True)

In [7]:
# Params work transparently in arithmetic, no unwrapping needed
x_sample = jnp.ones((1,))
result = x_sample @ model.fc_1.w

print(f"Result shape: {result.shape}")
print(f"Result type:  {type(result).__name__}  (regular array, not Param)")

Result shape: (16,)
Result type:  ArrayImpl  (regular array, not Param)


`.params` returns the full model pytree with non-`Param` leaves set to `None`. The tree structure is preserved, including static fields like activation functions.

In [8]:
model.params

MLP(
  fc_1=Linear(
    w=Param(f32[1, 16], trainable=True),
    b=Param(f32[16], trainable=True),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=True),
    b=Param(f32[1], trainable=True),
    activation=None,
  ),
  input_scale=None,
)

In [9]:
print(f"Total parameters: {model.num_params:,}")

Total parameters: 49


## Modules are pytrees

Ion modules are registered as JAX pytrees, so standard `jax.tree` operations work out of the box.

In [10]:
# The leaves of the tree are Param values and plain arrays
leaves = jax.tree.leaves(model)
print(f"{len(leaves)} leaves:")
for leaf in leaves:
    print(f"  {str(leaf.dtype):8s} {str(leaf.shape)}")

5 leaves:
  float32  (1, 16)
  float32  (16,)
  float32  (16, 1)
  float32  (1,)
  float32  ()


In [11]:
# Zero all leaves
jax.tree.map(lambda p: jnp.zeros_like(p), model)

MLP(
  fc_1=Linear(
    w=Param(f32[1, 16], trainable=True),
    b=Param(f32[16], trainable=True),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=True),
    b=Param(f32[1], trainable=True),
    activation=None,
  ),
  input_scale=f32[],
)

## JAX transforms just work

Because modules are standard pytrees, all native JAX transforms work directly with no wrappers. This includes `jax.jit`, `jax.vmap`, `jax.grad`, and higher-order transforms like `jax.jacobian` and `jax.hessian`.

In [12]:
# JIT-compile the model directly, it's just a callable pytree
jit_model = jax.jit(model)
y_hat = jit_model(x_train[:5])
print(f"JIT output shape: {y_hat.shape}")

JIT output shape: (5, 1)


In [13]:
# vmap over a batch dimension, also just works
batched_input = jax.random.normal(jax.random.key(1), (8, 1))
y_batched = jax.vmap(model)(batched_input)
print(f"vmap output shape: {y_batched.shape}")

vmap output shape: (8, 1)


### Gradients

`jax.grad` and `jax.value_and_grad` are no exception. The returned gradient tree has the same structure as the model. Frozen `Param` positions contain zeros thanks to `stop_gradient`, and `Optimizer.update` only modifies trainable `Param` leaves, so everything else passes through unchanged.

Note that non-`Param` array leaves (like `input_scale`) also receive real gradients from `jax.grad`, since they're part of the pytree. This can be useful for inspecting how the loss depends on any array in your model. `Optimizer.update` simply ignores them since they aren't parameters.

In [14]:
def loss_fn(model, x, y):
    y_hat = model(x)
    return jnp.mean((y_hat - y) ** 2)

loss, grads = jax.value_and_grad(loss_fn)(model, x_train, y_train)
print(f"Loss: {loss:.4f}")

Loss: 0.7954


In [15]:
# Same structure as the model, with gradients at each leaf
grads

MLP(
  fc_1=Linear(
    w=Param(f32[1, 16], trainable=True),
    b=Param(f32[16], trainable=True),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=True),
    b=Param(f32[1], trainable=True),
    activation=None,
  ),
  input_scale=f32[],
)

## Optimizer

`ion.Optimizer` wraps an [Optax](https://github.com/google-deepmind/optax) transform with Param-aware updates. It's registered as a JAX pytree itself, so it flows through `jax.jit` and `jax.lax.scan` like any other value.

In [16]:
optimizer = ion.Optimizer(optax.adam(5e-3), model)
optimizer

Optimizer(step=0, state_leaves=9)

### Auto-partitioning frozen params

When a model has frozen `Param` leaves, the optimizer automatically partitions the transform: trainable params get the real optimizer, frozen params get `optax.set_to_zero()`. This means no momentum or variance buffers are allocated for frozen weights.

In [17]:
# Freeze the first layer and compare optimizer state sizes
frozen_model = model.replace(fc_1=model.fc_1.freeze())
frozen_opt = ion.Optimizer(optax.adam(5e-3), frozen_model)

full_leaves = len(jax.tree.leaves(optimizer.state))
frozen_leaves = len(jax.tree.leaves(frozen_opt.state))
print(f"All trainable:  {full_leaves} state leaves")
print(f"fc_1 frozen:    {frozen_leaves} state leaves")

All trainable:  9 state leaves
fc_1 frozen:    5 state leaves


In [18]:
frozen_opt

Optimizer(step=0, state_leaves=5)

### Per-field transforms

Pass a dict to use different transforms for different parts of the model. Each key is a field name (or tuple of field names), each value is an optax transform. Useful for GANs, actor-critic RL, or transfer learning where different components need different learning rates or clipping.

Usage stays the same: one `jax.grad` call, one `optimizer.update`. The routing happens internally.

In [ ]:
# A simple two-headed model
class TwoHead(nn.Module):
    head_a: nn.Linear
    head_b: nn.Linear

    def __init__(self, *, key):
        keys = jax.random.split(key)
        self.head_a = nn.Linear(1, 8, key=keys[0])
        self.head_b = nn.Linear(1, 8, key=keys[1])

    def __call__(self, x):
        return self.head_a(x) + self.head_b(x)

two_head = TwoHead(key=jax.random.key(0))

# head_a and head_b get separate optimizers
optimizer = ion.Optimizer(
    {"head_a": optax.adam(1e-4), "head_b": optax.adam(4e-4)},
    two_head,
)

# Same grad + update pattern as before
grads = jax.grad(lambda m: jnp.sum(m(jnp.ones((1, 1)))))(two_head)
two_head, optimizer = optimizer.update(two_head, grads)
optimizer

Optimizer(step=1, state_leaves=10, fields=['head_a', 'head_b'])

## Convenience methods

`nn.Module` provides a few convenience methods that wrap the functional API for syntactic sugar. The functional forms (`ion.freeze`, `ion.unfreeze`, `ion.astype`) work on any pytree, while the methods are shorthand for the common case of performing them on a module.

### Freeze and unfreeze

You can freeze an entire model or individual layers. Frozen parameters have `trainable=False`, which causes `stop_gradient` to be applied automatically via `__jax_array__`. This means `jax.grad` naturally produces zero gradients for frozen parameters, and XLA skips the backward computation through them entirely.

In [20]:
# Freeze the entire model
frozen_model = model.freeze()
frozen_model

MLP(
  fc_1=Linear(
    w=Param(f32[1, 16], trainable=False),
    b=Param(f32[16], trainable=False),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=False),
    b=Param(f32[1], trainable=False),
    activation=None,
  ),
  input_scale=f32[],
)

In [21]:
# Selectively freeze just the first layer
partially_frozen = model.replace(fc_1=model.fc_1.freeze())
partially_frozen

MLP(
  fc_1=Linear(
    w=Param(f32[1, 16], trainable=False),
    b=Param(f32[16], trainable=False),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=True),
    b=Param(f32[1], trainable=True),
    activation=None,
  ),
  input_scale=f32[],
)

In [22]:
# Frozen params produce zero gradients (stop_gradient is applied automatically)
frozen_grads = jax.grad(loss_fn)(partially_frozen, x_train, y_train)
frozen_grads

MLP(
  fc_1=Linear(
    w=Param(f32[1, 16], trainable=False),
    b=Param(f32[16], trainable=False),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(f32[16, 1], trainable=True),
    b=Param(f32[1], trainable=True),
    activation=None,
  ),
  input_scale=f32[],
)

### Dtype casting

`model.astype()` casts all matching leaves to a target dtype. Float targets cast float leaves, int targets cast int leaves, and so on.

In [23]:
# Cast all float params to bfloat16
bf16_model = model.astype(jnp.bfloat16)
bf16_model

MLP(
  fc_1=Linear(
    w=Param(bf16[1, 16], trainable=True),
    b=Param(bf16[16], trainable=True),
    activation=relu,
  ),
  fc_2=Linear(
    w=Param(bf16[16, 1], trainable=True),
    b=Param(bf16[1], trainable=True),
    activation=None,
  ),
  input_scale=bf16[],
)

## Training loop

Putting it together: compute gradients with `jax.value_and_grad`, then apply them with `optimizer.update`.

In [24]:
# Start fresh
model = MLP(in_dim=1, hidden_dim=16, out_dim=1, key=jax.random.key(0))

optimizer = ion.Optimizer(optax.adam(5e-3), model)


@jax.jit
def train_step(model, optimizer, x, y):
    loss, grads = jax.value_and_grad(loss_fn)(model, x, y)
    model, optimizer = optimizer.update(model, grads)
    return model, optimizer, loss


losses = []
for step in range(200):
    model, optimizer, loss = train_step(model, optimizer, x_train, y_train)
    losses.append(loss.item())

print(f"Final loss: {losses[-1]:.6f}")

Final loss: 0.045831


In [25]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=losses, mode="lines", name="MSE loss", line=dict(width=5, color="#636EFA"), opacity=0.8))
fig.update_layout(title="Training loss", xaxis_title="Step", yaxis_title="Loss", height=400, template="plotly_white", yaxis_type="log")
fig.show()

In [26]:
x_eval = jnp.linspace(-jnp.pi, jnp.pi, 500).reshape(-1, 1)
x_sin = jnp.linspace(-jnp.pi, jnp.pi, 100)
y_pred = model(x_eval)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_sin, y=jnp.sin(x_sin), mode="lines", name="sin(x)", line=dict(width=2, color="rgb(96,96,96)", dash="dash")))
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=y_pred[:, 0], mode="lines", name="Model", line=dict(width=5, color="#636EFA"), opacity=0.85))
fig.update_layout(title="Trained model vs sin(x)", xaxis_title="x", yaxis_title="y", height=400, template="plotly_white")
fig.show()

## Model surgery

Modules are immutable after `__init__`, so you can't mutate them in place. Use `.replace()` to create a modified copy.

In [27]:
# Direct mutation is not allowed
try:
    model.fc_1 = Linear(1, 64, key=jax.random.key(99))
except AttributeError as e:
    print(f"AttributeError: {e}")

AttributeError: Cannot set attribute 'fc_1', MLP is frozen after __init__.


### Swap activation function

`activation` lives on the `Linear` layer and can be swapped with nested `.replace()` calls. Since it's static metadata, `jax.jit` handles the recompilation automatically.

In [28]:
# Swap activation on the first layer
tanh_model = model.replace(fc_1=model.fc_1.replace(activation=jax.nn.tanh))
opt_state_tanh = ion.Optimizer(optax.adam(5e-3), tanh_model)

tanh_losses = []
for step in range(200):
    tanh_model, opt_state_tanh, loss = train_step(tanh_model, opt_state_tanh, x_train, y_train)
    tanh_losses.append(loss.item())

print(f"ReLU final loss: {losses[-1]:.6f}")
print(f"Tanh final loss: {tanh_losses[-1]:.6f}")

ReLU final loss: 0.045831
Tanh final loss: 0.163851


In [29]:
y_pred_tanh = tanh_model(x_eval)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_sin, y=jnp.sin(x_sin), mode="lines", name="sin(x)", line=dict(width=2, color="rgb(96,96,96)", dash="dash")))
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=y_pred[:, 0], mode="lines", name="ReLU", line=dict(width=5, color="#636EFA"), opacity=0.85))
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=y_pred_tanh[:, 0], mode="lines", name="Tanh", line=dict(width=5, color="#EF553B"), opacity=0.85))
fig.update_layout(title="Activation function comparison", xaxis_title="x", yaxis_title="y", height=400, template="plotly_white")
fig.show()

### Resize hidden layers

We can swap in wider layers with `.replace()`. The weights are reinitialized since the dimensions change.

In [30]:
keys = jax.random.split(jax.random.key(42))
wide_model = model.replace(
    fc_1=Linear(1, 128, activation=jax.nn.relu, key=keys[0]),
    fc_2=Linear(128, 1, key=keys[1]),
)

print(f"Original:  {model.num_params:,} params  (hidden_dim=16)")
print(f"Wide:     {wide_model.num_params:,} params  (hidden_dim=128)")

Original:  49 params  (hidden_dim=16)
Wide:     385 params  (hidden_dim=128)


In [31]:
opt_state_wide = ion.Optimizer(optax.adam(5e-3), wide_model)

wide_losses = []
for step in range(200):
    wide_model, opt_state_wide, loss = train_step(wide_model, opt_state_wide, x_train, y_train)
    wide_losses.append(loss.item())

y_pred_wide = wide_model(x_eval)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_sin, y=jnp.sin(x_sin), mode="lines", name="sin(x)", line=dict(width=2, color="rgb(96,96,96)", dash="dash")))
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=y_pred[:, 0], mode="lines", name="16 hidden", line=dict(width=5, color="#636EFA"), opacity=0.85))
fig.add_trace(go.Scatter(x=x_eval[:, 0], y=y_pred_wide[:, 0], mode="lines", name="128 hidden", line=dict(width=5, color="#00CC96"), opacity=0.85))
fig.update_layout(title="Effect of hidden size", xaxis_title="x", yaxis_title="y", height=400, template="plotly_white")
fig.show()

## Checkpointing

`ion.save()` serializes array leaves and trainable flags to a `.npz` file. `ion.load()` restores them into an existing model structure, which provides the non-array fields like activation functions.

In [32]:
import os
import tempfile

path = os.path.join(tempfile.gettempdir(), "model.npz")
ion.save(path, model)
print(f"Saved to {path}  ({os.path.getsize(path):,} bytes)")

Saved to /var/folders/zv/91s6yg812kz5b5j9q8ns1rb80000gn/T/model.npz  (1,872 bytes)


In [33]:
# Load into a fresh (randomly initialized) model
fresh_model = MLP(in_dim=1, hidden_dim=16, out_dim=1, key=jax.random.key(999))

print(f"Fresh model loss:  {loss_fn(fresh_model, x_train, y_train):.4f}")

loaded_model = ion.load(path, fresh_model)
print(f"Loaded model loss: {loss_fn(loaded_model, x_train, y_train):.6f}")
print(f"Weights match:     {jnp.allclose(model.fc_1.w._value, loaded_model.fc_1.w._value)}")

Fresh model loss:  0.4325
Loaded model loss: 0.045351
Weights match:     True


In [34]:
# Trainable flags are preserved through save/load
frozen_model = model.replace(fc_1=model.fc_1.freeze())
ion.save(path, frozen_model)

loaded_frozen = ion.load(path, fresh_model)
print(f"fc_1.w trainable: {loaded_frozen.fc_1.w.trainable}  (was frozen before save)")
print(f"fc_2.w trainable: {loaded_frozen.fc_2.w.trainable}  (was trainable before save)")

os.remove(path)

fc_1.w trainable: False  (was frozen before save)
fc_2.w trainable: True  (was trainable before save)


## Higher-order derivatives

We've used `jax.grad` to train models. But JAX also has higher-order transforms like `jax.jacobian` and `jax.hessian` that work directly on Ion models, no wrappers needed.

To show this, we'll train a 2-output model that learns $[\sin(x),\, \cos(x)]$ simultaneously, then use the Jacobian and Hessian to extract learned derivatives of $\cos(x)$.

In [35]:
class MultiOutputMLP(nn.Module):
    fc_1: Linear
    fc_2: Linear

    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, *, key: PRNGKeyArray):
        keys = jax.random.split(key)
        self.fc_1 = Linear(in_dim, hidden_dim, activation=jax.nn.relu, key=keys[0])
        self.fc_2 = Linear(hidden_dim, out_dim, key=keys[1])

    def __call__(self, x):
        return self.fc_2(self.fc_1(x))


# Train on [sin(x), cos(x)]
y_target = jnp.stack([jnp.sin(x_train[:, 0]), jnp.cos(x_train[:, 0])], axis=-1)

model = MultiOutputMLP(in_dim=1, hidden_dim=64, out_dim=2, key=jax.random.key(7))
optimizer = ion.Optimizer(optax.adam(5e-3), model)

for step in range(500):
    model, optimizer, loss = train_step(model, optimizer, x_train, y_target)

print(f"Final loss: {loss:.6f}")

Final loss: 0.000341


### Jacobian and Hessian

`jax.jacobian` differentiates each output with respect to each input. Our model maps $\mathbb{R}^1 \to \mathbb{R}^2$, so the Jacobian at each point is a $2 \times 1$ matrix. The Hessian gives second derivatives. Both are computed in one line each.

We'll focus on the $\cos(x)$ output and overlay the analytical derivatives for comparison. Since ReLU is piecewise linear, the Jacobian is piecewise constant and the Hessian is zero everywhere.

In [36]:
y_pred = model(x_eval)
x_plot = x_eval[:, 0]

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_plot, y=jnp.sin(x_plot), mode="lines", name="sin(x)", line=dict(width=2, color="rgb(96,96,96)", dash="dash")))
fig.add_trace(go.Scatter(x=x_plot, y=jnp.cos(x_plot), mode="lines", name="cos(x)", line=dict(width=2, color="rgb(96,96,96)", dash="dash")))
fig.add_trace(go.Scatter(x=x_plot, y=y_pred[:, 0], mode="lines", name="learned sin", line=dict(width=5, color="#636EFA"), opacity=0.15))
fig.add_trace(go.Scatter(x=x_plot, y=y_pred[:, 1], mode="lines", name="learned cos", line=dict(width=5, color="#636EFA"), opacity=0.85))
fig.update_layout(title="Model outputs", xaxis_title="x", yaxis_title="y", height=400, template="plotly_white")
fig.show()

In [37]:
jac = jax.vmap(jax.jacobian(model))(x_eval)  # (500, 2, 1)
hess = jax.vmap(jax.hessian(model))(x_eval)  # (500, 2, 1, 1)

# Focus on cos output (index 1)
y_cos = y_pred[:, 1]
dy = jac[:, 1, 0]
d2y = hess[:, 1, 0, 0]

fig = go.Figure()
# Analytical references
fig.add_trace(go.Scatter(x=x_plot, y=jnp.cos(x_plot), mode="lines", name="cos(x)", line=dict(width=2, color="#636EFA", dash="dash"), opacity=0.5))
fig.add_trace(go.Scatter(x=x_plot, y=-jnp.sin(x_plot), mode="lines", name="-sin(x)", line=dict(width=2, color="#AB63FA", dash="dash"), opacity=0.5))
fig.add_trace(go.Scatter(x=x_plot, y=-jnp.cos(x_plot), mode="lines", name="-cos(x)", line=dict(width=2, color="#EF553B", dash="dash"), opacity=0.5))
# Learned
fig.add_trace(go.Scatter(x=x_plot, y=y_cos, mode="lines", name="y", line=dict(width=5, color="#636EFA"), opacity=0.85))
fig.add_trace(go.Scatter(x=x_plot, y=dy, mode="lines", name="dy/dx", line=dict(width=5, color="#AB63FA"), opacity=0.85))
fig.add_trace(go.Scatter(x=x_plot, y=d2y, mode="lines", name="d\u00b2y/dx\u00b2", line=dict(width=5, color="#EF553B"), opacity=0.85))
fig.update_layout(title="Learned derivatives of cos(x) (ReLU)", xaxis_title="x", yaxis_title="value", height=400, template="plotly_white")
fig.show()

### Surgery for smooth derivatives

The Jacobian is a staircase and the Hessian is flat zero because ReLU is piecewise linear. Swap to tanh with `.replace()` and retrain to get smooth derivatives that match the analytical curves.

In [38]:
# One-line surgery: swap activation from relu to tanh
model_tanh = model.replace(fc_1=model.fc_1.replace(activation=jax.nn.tanh))
opt_tanh = ion.Optimizer(optax.adam(5e-3), model_tanh)

for step in range(500):
    model_tanh, opt_tanh, loss = train_step(model_tanh, opt_tanh, x_train, y_target)

print(f"Final loss (tanh): {loss:.6f}")

Final loss (tanh): 0.002407


In [39]:
y_pred_tanh = model_tanh(x_eval)

jac_tanh = jax.vmap(jax.jacobian(model_tanh))(x_eval)
hess_tanh = jax.vmap(jax.hessian(model_tanh))(x_eval)

# Focus on cos output (index 1)
y_cos_tanh = y_pred_tanh[:, 1]
dy_tanh = jac_tanh[:, 1, 0]
d2y_tanh = hess_tanh[:, 1, 0, 0]

fig = go.Figure()
# Analytical references
fig.add_trace(go.Scatter(x=x_plot, y=jnp.cos(x_plot), mode="lines", name="cos(x)", line=dict(width=2, color="#636EFA", dash="dash"), opacity=0.5))
fig.add_trace(go.Scatter(x=x_plot, y=-jnp.sin(x_plot), mode="lines", name="-sin(x)", line=dict(width=2, color="#AB63FA", dash="dash"), opacity=0.5))
fig.add_trace(go.Scatter(x=x_plot, y=-jnp.cos(x_plot), mode="lines", name="-cos(x)", line=dict(width=2, color="#EF553B", dash="dash"), opacity=0.5))
# Learned
fig.add_trace(go.Scatter(x=x_plot, y=y_cos_tanh, mode="lines", name="y", line=dict(width=5, color="#636EFA"), opacity=0.85))
fig.add_trace(go.Scatter(x=x_plot, y=dy_tanh, mode="lines", name="dy/dx", line=dict(width=5, color="#AB63FA"), opacity=0.85))
fig.add_trace(go.Scatter(x=x_plot, y=d2y_tanh, mode="lines", name="d\u00b2y/dx\u00b2", line=dict(width=5, color="#EF553B"), opacity=0.85))
fig.update_layout(title="Learned derivatives of cos(x) (tanh)", xaxis_title="x", yaxis_title="value", height=400, template="plotly_white")
fig.show()

---

That covers Ion's core. Ion ships with standard layers (linear, convolution, attention, normalization, recurrent, pooling, and more) built on these same concepts. See the [README](https://github.com/auxeno/ion#layers) for the full list. For a complete training example, see [cnn_mnist.py](https://github.com/auxeno/ion/blob/main/examples/cnn_mnist.py).